# Geneformer In-Silico Perturbation (ISP) Pipeline
### Based on Liu et al. — *Evaluating Foundation Models for In-Silico Perturbation*

---

**Pipeline overview:**
1. Install dependencies
2. Load and QC `.h5ad` files
3. Auto-annotate cell types (CellTypist) → split into per-cell-type adatas
4–13. **Per-cell-type loop** (gene mapping → tokenize → embeddings → separation test → ISP → stats → figures → enrichment)
14. Cross-cell-type summary plots (dot plots, comparison figures)
15. Export final results

Make sure all file directories are appropriately updated



## 0. Install Dependencies

In [ ]:
# Core single-cell stack
!pip install -q anndata scanpy scipy pandas numpy matplotlib seaborn

# Geneformer (from HuggingFace / Theodoris lab)
!pip install -q transformers datasets
!pip install -q pyarrow

# loompy for writing loom files (anndata.write_loom is deprecated)
!pip install -q loompy

# Cell type annotation
!pip install -q celltypist

# Gene ID mapping
!pip install -q mygene

# Pathway enrichment
!pip install -q gseapy

# Stats
!pip install -q statsmodels scikit-learn

# Clone Geneformer repo (for tokenizer + ISP utilities)
#set appropriate path
import os
if not os.path.exists('/content/Geneformer'):
    !git clone https://huggingface.co/ctheodoris/Geneformer /content/Geneformer

# Install Geneformer package (separate step to avoid %cd issues)
!pip install -q -e /content/Geneformer

# Add to path
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

print('✅ All dependencies installed.')

In [ ]:
!pip install "transformers==4.57.1"
!pip install .

In [ ]:
# ensure datasets updated in Colab environment
!pip install gcsfs==2025.3.0
!pip install datasets==3.6.0

## 1. Configuration — Set Parameters Here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#not needed if not using Colab

In [ ]:
#set
import os
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')
import scanpy as sc

H5AD_PATHS = [
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE159977.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE185477.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE189600.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE190487.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE192740.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE202379.h5ad'
]
file1= sc.read_h5ad('/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE159977.h5ad')
file2= sc.read_h5ad('/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE185477.h5ad')
file3= sc.read_h5ad('/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE189600.h5ad')
file4= sc.read_h5ad('/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE190487.h5ad')
file5= sc.read_h5ad('/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE192740.h5ad')
file6= sc.read_h5ad('/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE202379.h5ad')

import anndata as ad
mergedliver = ad.concat([file1, file2,file3,file6])
mergedimmune = ad.concat([file4, file5])
mergedliver.write('/content/drive/MyDrive/scFM perturbation project/input_scFM/mergedliver.h5ad')
mergedimmune.write('/content/drive/MyDrive/scFM perturbation project/input_scFM/mergedimmune.h5ad')
H5AD_PATHS=[
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/mergedliver.h5ad'#,
    #'/content/drive/MyDrive/scFM perturbation project/input_scFM/mergedimmune.h5ad'
]

CONDITION_COL  = None
CONTROL_LABEL  = 'Healthy'
TARGET_LABEL   = 'MASH'
ALT_STATES = []
CELLTYPE_COL   = None
FOCUS_CELLTYPES = ['CD16- NK cells', 'Tem/Trm cytotoxic T cells', 'CD16+ NK cells', 'MAIT cells',
                   'Tem/Effector helper T cells', 'NK cells', 'Tem/Temra cytotoxic T cells',
                   'Tcm/Naive helper T cells', 'Naive B cells', 'CRTAM+ gamma-delta T cells',
                   'Hepatocytes', 'T cells', 'Endothelial cells', 'Fibroblasts', 'Macrophages',
                   'Cholangiocytes', 'Mono+mono derived cells', 'B cells', 'Plasma cells',
                   'Memory B cells', 'DC2', 'Classical monocytes', 'pDC',
                   'Intestinal macrophages', 'Non-classical monocytes', 'Resident NK',
                   'Circulating NK/NKT', 'Neutrophils']

DONOR_COL = 'patient_id'

MAX_ISP_CELLS_TOTAL     = None
MIN_CELLS_PER_DONOR_ISP = 5

SAVED_EMBEDDINGS_PATH = None

GENE_ID_TYPE   = 'symbol'
SPECIES        = 'human'
PERTURB_MODE   = 'down'
N_CANDIDATE_GENES = 200
MIN_CELLS_PER_STATE = 100
OUTPUT_DIR = '/content/geneformer_isp_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration set.')
print(f'   Control        : {CONTROL_LABEL}')
print(f'   Target         : {TARGET_LABEL}')
print(f'   Perturb        : {PERTURB_MODE}-regulation')
print(f'   Donor col      : {DONOR_COL}')
print(f'   Saved embeddings: {SAVED_EMBEDDINGS_PATH}')

In [ ]:
# Auto-detect donor column from common column names
_DONOR_CANDIDATES = [
    'donor', 'donor_id', 'patient', 'patient_id', 'sample', 'sample_id',
    'subject', 'subject_id', 'individual', 'participant', 'Donor', 'Patient',
    'SampleID', 'orig.ident', 'batch',
]

def detect_donor_col(adata, override=None):
    """Return the first matching donor column found in adata.obs."""
    if override is not None and override is not False:
        if override in adata.obs.columns:
            return override
        print(f'   ⚠️  DONOR_COL="{override}" not found in obs.')
        return None
    if override is False:
        return None
    for c in _DONOR_CANDIDATES:
        if c in adata.obs.columns:
            print(f'   Auto-detected donor column: "{c}"')
            return c
    print('   ⚠️  No donor column detected — patient-aware sampling will fall back to random.')
    return None

print('✅ Donor detection helper ready.')

## 2. Load & QC AnnData Objects

In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

def detect_condition_col(adata):
    candidate = ['broad_condition']
    for c in candidate:
        if c in adata.obs.columns:
            print(f'   Auto-detected condition column: "{c}"')
            print(f'   Values: {adata.obs[c].unique().tolist()}')
            return c
    return None

def load_and_qc(path, condition_col_override=None):
    print(f'\n📂 Loading: {path}')
    adata = sc.read_h5ad(path)
    print(f'   Shape  : {adata.shape[0]:,} cells × {adata.shape[1]:,} genes')
    print(f'   obs    : {list(adata.obs.columns)}')
    if adata.var_names.duplicated().any():
        print('   ⚠️  Duplicate gene names detected — making unique.')
        adata.var_names_make_unique()
    if 'counts' not in adata.layers:
        X = adata.X
        if sp.issparse(X):
            vals = X.data[:1000] if X.nnz > 1000 else X.data
        else:
            vals = np.asarray(X).flat[:1000]
        if np.allclose(vals, np.round(vals), atol=1e-3):
            print('   ✅ .X appears to be raw counts — copying to layers["counts"].')
            adata.layers['counts'] = adata.X.copy()
        else:
            print('   ⚠️  .X may be normalized. Checking adata.raw...')
            if adata.raw is not None:
                print('   ✅ Found adata.raw — using as raw counts.')
                raw_adata = adata.raw.to_adata()
                adata.layers['counts'] = raw_adata[:, adata.var_names].X.copy()
            else:
                print('   ❌ Cannot confirm raw counts. Proceeding with .X — verify manually!')
                adata.layers['counts'] = adata.X.copy()
    mito_prefix = 'MT-' if SPECIES == 'human' else 'mt-'
    adata.var['mt'] = adata.var_names.str.startswith(mito_prefix)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    adata = adata[adata.obs['n_genes_by_counts'] > 200].copy()
    adata = adata[adata.obs['pct_counts_mt'] < 20].copy()
    sc.pp.filter_genes(adata, min_cells=10)
    n_after = adata.n_obs
    print(f'   QC: {n_before:,} → {n_after:,} cells retained ({n_before - n_after:,} removed)')
    global CONDITION_COL
    if condition_col_override:
        adata.obs['_condition'] = adata.obs[condition_col_override].astype(str)
    elif CONDITION_COL and CONDITION_COL in adata.obs.columns:
        adata.obs['_condition'] = adata.obs[CONDITION_COL].astype(str)
    else:
        detected = detect_condition_col(adata)
        if detected:
            CONDITION_COL = detected
            adata.obs['_condition'] = adata.obs[detected].astype(str)
        else:
            print('   ❌ Could not detect condition column. Set CONDITION_COL manually.')
            adata.obs['_condition'] = 'unknown'
    print(f'   Conditions: {adata.obs["_condition"].value_counts().to_dict()}')
    return adata

adatas = []
for path in H5AD_PATHS:
    if os.path.exists(path):
        adatas.append(load_and_qc(path))
    else:
        print(f'⚠️  File not found: {path}')
if not adatas:
    raise FileNotFoundError('No .h5ad files loaded. Check H5AD_PATHS.')
print(f'\n✅ Loaded {len(adatas)} dataset(s).')

## 3. Cell Type Annotation
Uses **CellTypist** (pretrained on 36 human tissues) if cell type labels are not already present.
At the end of this step, `adata_by_celltype` is a dict mapping each cell type name to its own AnnData object.

In [ ]:
# for adata in adatas:
#     if adata.obs.index.duplicated().any():
#         print(adata)
#         print(f"Duplicate cells before: {adata.obs.index.duplicated().sum()}")
#         adata = adata[~adata.obs.index.duplicated(keep='first')]
#         print(f"Duplicate cells after: {adata.obs.index.duplicated().sum()}")

In [ ]:
import celltypist
from celltypist import models
import scipy.sparse as sp

liver_gses = ['GSE212837', 'GSE189600', 'GSE174748', 'GSE192740', 'GSE185477', 'GSE202379','GSE270488', 'GSE159977', 'GSE190487', 'GSE192740']

def get_celltypist_model_for_file(filename):
    return 'Healthy_Human_Liver.pkl'

def annotate_cell_types(adata, use_celltypist=True, filename=None):
    global CELLTYPE_COL
    ct_candidates = ['cell_type','celltype','CellType','cell_ontology_class','Celltype','leiden_celltypes','anno','annotation','cluster_annotation','broad_celltypes']
    if CELLTYPE_COL and CELLTYPE_COL in adata.obs.columns:
        adata.obs['_cell_type'] = adata.obs[CELLTYPE_COL].astype(str)
        print(f'   Using existing cell type column: "{CELLTYPE_COL}"')
        print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
        return adata
    for c in ct_candidates:
        if c in adata.obs.columns:
            CELLTYPE_COL = c
            adata.obs['_cell_type'] = adata.obs[c].astype(str)
            print(f'   Auto-detected cell type column: "{c}"')
            print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
            return adata
    print('   No cell type labels found — running CellTypist annotation...')
    model_name = get_celltypist_model_for_file(filename)
    adata_ct = adata.copy()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)
    try:
        model = models.Model.load(model=model_name)
        try:
            predictions = celltypist.annotate(adata_ct, model=model, majority_voting=True)
            adata1 = predictions.to_adata()
            adata1 = adata1[adata1.obs.conf_score > 0.97]
            adata1.obs['_cell_type'] = adata1.obs.majority_voting
            adata = adata1.copy()
        except Exception:
            predictions = celltypist.annotate(adata_ct, model=model, majority_voting=False)
            adata1 = predictions.to_adata()
            adata1 = adata1[adata1.obs.conf_score > 0.97]
            adata1.obs['_cell_type'] = adata1.obs.predicted_labels
            adata = adata1.copy()
            print(f'   ⚠️  Majority voting failed, using per-cell predictions instead.')
        print(f'   ✅ CellTypist annotation complete ({model_name}).')
    except Exception as e:
        adata.obs['_cell_type'] = 'Unknown'
        print(f'   ❌ CellTypist failed ({e}). All cells labelled "Unknown".')
    print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
    return adata

for i, adata in enumerate(adatas):
    print(f'\n--- Dataset {i+1} ---')
    adatas[i] = annotate_cell_types(adata, filename=H5AD_PATHS[i])
print('\n✅ Cell type annotation complete.')

combine = ad.concat(adatas)
combine.write('/content/drive/MyDrive/scFM perturbation project/input_scFM/combined_data.h5ad')

# ── Build per-cell-type adata dict (the key v5 change) ──────────────────────
# adata_by_celltype maps each cell type string → its subset AnnData.
# Only cell types in FOCUS_CELLTYPES are kept (if FOCUS_CELLTYPES is non-empty).
adata_by_celltype = {}
for celltype in combine.obs['_cell_type'].unique():
    if FOCUS_CELLTYPES and celltype not in FOCUS_CELLTYPES:
        continue
    adata_by_celltype[celltype] = combine[combine.obs['_cell_type'] == celltype].copy()

print(f'\n✅ Built adata_by_celltype: {len(adata_by_celltype)} cell types.')
for ct, a in adata_by_celltype.items():
    print(f'   {ct}: {a.n_obs:,} cells')

## 4–13. Per-Cell-Type Loop

Each cell type runs through:
- Gene mapping → tokenization → embedding extraction → separation test
- If separation test **fails** → cell type is skipped for ISP but recorded
- If separation test **passes** → ISP → stats → figures → enrichment

All per-cell-type results are collected into `all_results` and `separation_results_all`
for the cross-cell-type summary plots in Step 14.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SHARED IMPORTS & ONE-TIME SETUP
# (run once before the per-cell-type loop)
# ════════════════════════════════════════════════════════════════════════════

import sys, os, time, hashlib, shutil, tempfile, subprocess, pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import loompy
import datasets as hf_datasets
import mygene
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from collections import defaultdict
from scipy import stats
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import gseapy as gp

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8,
})

if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

from geneformer import TranscriptomeTokenizer, EmbExtractor, InSilicoPerturber
import geneformer.perturber_utils as pu

mg = mygene.MyGeneInfo()

# GPU
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['TOKENIZERS_PARALLELISM']   = 'false'
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU — ISP will be slow.')

# Patch pu.load_model for GPU (once per session)
if not getattr(pu, '_load_model_gpu_patched', False):
    _REAL_LOAD_MODEL = pu.load_model
    def _gpu_load_model(model_type, num_classes, model_directory, *args, **kwargs):
        model = _REAL_LOAD_MODEL(model_type, num_classes, model_directory, *args, **kwargs)
        if torch.cuda.is_available():
            model = model.to('cuda:0').eval()
            print(f'   Model device: {next(model.parameters()).device}')
        return model
    pu.load_model = _gpu_load_model
    pu._load_model_gpu_patched = True
    print('✅ pu.load_model patched for GPU.')

# Constants
MODE_MAP        = {'down': 'delete', 'delete': 'delete', 'up': 'overexpress'}
gf_perturb_type = MODE_MAP.get(PERTURB_MODE, 'delete')
MIN_CELLS_PER_GENE    = 20
EFFECT_SIZE_THRESHOLD = 2.5e-4
N_BOOTSTRAP_SPLITS    = 10
SPLIT_FRACTION        = 0.5
MIN_CELLS_HALF        = 5
_alt_states           = ALT_STATES if 'ALT_STATES' in dir() and ALT_STATES else []
states_to_run         = [TARGET_LABEL] + list(_alt_states)

# Colour scheme
SIG_GREEN         = '#2ecc71'
FAIL_RED          = '#e74c3c'
SIG_COLOR         = '#C0392B'
NSIG_COLOR        = '#BDC3C7'
ALT_COLOR         = '#2980B9'
PAN_DISEASE_COLOR = '#8E44AD'
GOAL_LIGHT        = '#E8A49C'
ALT_LIGHT         = '#A9C4D9'
AMBER             = '#F5A623'

# Accumulators (filled during the loop, consumed in Step 14)
all_results            = []   # list of scored DataFrames
all_sep_rows           = []   # list of separation test summary rows
separation_results_all = {}   # {cell_type: {comp_name: result_dict}}
passing_celltypes      = []   # cell types that passed all sep tests
embeddings_all         = []   # list of per-cell-type embedding DataFrames

print('✅ Shared setup complete.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MASH / CANDIDATE GENE LIST  (updated with genes from Elison et al. preprint supplementary table 8 and Hong et al. 2025 supplementary table 11)
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/scFM perturbation project/candidate_gene.csv')
MASH_GENES = df['Gene'].tolist()
_seen = set()
_deduped = []
for g in MASH_GENES:
    if g not in _seen:
        _deduped.append(g)
        _seen.add(g)
MASH_GENES = _deduped
print(f'✅ {len(MASH_GENES)} unique MASH candidate genes loaded.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS  (separation test, ISP scoring, etc.)
# Defined once here; called inside the per-cell-type loop below.
# ════════════════════════════════════════════════════════════════════════════

primary_comp = f'{CONTROL_LABEL} vs {TARGET_LABEL}'
state_pairs  = [(CONTROL_LABEL, TARGET_LABEL, primary_comp)]
for alt_state in _alt_states:
    state_pairs.append((CONTROL_LABEL, alt_state, f'{CONTROL_LABEL} vs {alt_state}'))
    state_pairs.append((TARGET_LABEL,  alt_state, f'{TARGET_LABEL} vs {alt_state}'))


def run_separation_test_publishable(
        embeddings_df, cell_type, control_label, target_label,
        emb_cols=None, donor_col='donor'):
    ct_mask   = embeddings_df['cell_type'] == cell_type
    ctrl_mask = ct_mask & (embeddings_df['condition'] == control_label)
    tgt_mask  = ct_mask & (embeddings_df['condition'] == target_label)
    n_ctrl, n_tgt = ctrl_mask.sum(), tgt_mask.sum()
    empty = {'passes': False, 'p_value': 1.0, 'sep_score': 0.0,
             'cohens_d': None, 'balanced_accuracy': 0.5,
             'n_ctrl': n_ctrl, 'n_tgt': n_tgt,
             'n_donors_ctrl': 0, 'n_donors_tgt': 0,
             'ctrl_margins': np.array([]), 'tgt_margins': np.array([])}
    if n_ctrl < MIN_CELLS_PER_STATE or n_tgt < MIN_CELLS_PER_STATE:
        print(f'   {cell_type} [{control_label} vs {target_label}]: '
              f'insufficient cells (ctrl={n_ctrl}, tgt={n_tgt}). Skipping.')
        return empty
    if emb_cols is None:
        meta = {'cell_type', 'condition', 'donor', 'cell_id', 'index'}
        emb_cols = [c for c in embeddings_df.columns
                    if c not in meta and pd.api.types.is_numeric_dtype(embeddings_df[c])]
    ctrl_embs = embeddings_df.loc[ctrl_mask, emb_cols].values.astype(np.float32)
    tgt_embs  = embeddings_df.loc[tgt_mask,  emb_cols].values.astype(np.float32)
    ctrl_centroid = ctrl_embs.mean(axis=0, keepdims=True)
    tgt_centroid  = tgt_embs.mean(axis=0,  keepdims=True)
    ctrl_margins = (cosine_similarity(ctrl_embs, ctrl_centroid).flatten() -
                    cosine_similarity(ctrl_embs, tgt_centroid).flatten())
    tgt_margins  = (cosine_similarity(tgt_embs,  tgt_centroid).flatten() -
                    cosine_similarity(tgt_embs,  ctrl_centroid).flatten())
    _, p_ctrl = stats.mannwhitneyu(ctrl_margins, np.zeros(len(ctrl_margins)), alternative='greater')
    _, p_tgt  = stats.mannwhitneyu(tgt_margins,  np.zeros(len(tgt_margins)),  alternative='greater')
    sep_score  = (ctrl_margins.mean() + tgt_margins.mean()) / 2
    p_combined = max(p_ctrl, p_tgt)
    n_donors_ctrl = n_donors_tgt = 0
    cohens_d = None
    actual_donor_col = None
    for dc in ([donor_col] if donor_col else []) + ['donor', 'Donor', 'patient_id']:
        if dc and dc in embeddings_df.columns:
            actual_donor_col = dc
            break
    if actual_donor_col:
        ctrl_donors = embeddings_df.loc[ctrl_mask, actual_donor_col].values
        tgt_donors  = embeddings_df.loc[tgt_mask,  actual_donor_col].values
        n_donors_ctrl = len(np.unique(ctrl_donors))
        n_donors_tgt  = len(np.unique(tgt_donors))
        ctrl_donor_means = pd.Series(ctrl_margins, index=ctrl_donors).groupby(level=0).mean()
        tgt_donor_means  = pd.Series(tgt_margins,  index=tgt_donors).groupby(level=0).mean()
        if len(ctrl_donor_means) >= 2 and len(tgt_donor_means) >= 2:
            pooled_sd = np.sqrt((ctrl_donor_means.std()**2 + tgt_donor_means.std()**2) / 2 + 1e-10)
            cohens_d  = float((tgt_donor_means.mean() - ctrl_donor_means.mean()) / pooled_sd)
    X_all = np.vstack([ctrl_embs, tgt_embs])
    y_all = np.array([0]*len(ctrl_embs) + [1]*len(tgt_embs))
    ba_scores = []
    n_splits  = min(3, min(n_ctrl, n_tgt) // 10)
    if n_splits >= 2:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        for train_idx, test_idx in skf.split(X_all, y_all):
            clf = LogisticRegression(max_iter=500, C=0.1, random_state=42, solver='lbfgs')
            clf.fit(X_all[train_idx], y_all[train_idx])
            ba_scores.append(balanced_accuracy_score(y_all[test_idx], clf.predict(X_all[test_idx])))
    balanced_accuracy = float(np.mean(ba_scores)) if ba_scores else 0.5
    passes = (p_combined < 0.05) and (sep_score > 0)
    return {
        'passes': passes, 'p_value': float(p_combined), 'sep_score': float(sep_score),
        'cohens_d': cohens_d, 'balanced_accuracy': balanced_accuracy,
        'n_ctrl': int(n_ctrl), 'n_tgt': int(n_tgt),
        'n_donors_ctrl': int(n_donors_ctrl), 'n_donors_tgt': int(n_donors_tgt),
        'ctrl_margins': ctrl_margins, 'tgt_margins': tgt_margins,
    }


def all_comparisons_pass(ct_results):
    for label_a, label_b, comp_name in state_pairs:
        res = ct_results.get(comp_name, {})
        if res.get('n_ctrl', 0) == 0 or res.get('n_tgt', 0) == 0:
            return False
        if not res.get('passes', False):
            return False
    return True


def patient_aware_sample_for_isp(
        dataset, donor_key='donor',
        max_cells_total=None, min_cells_per_donor=5, random_state=42):
    """Donor-stratified sampling.

    When max_cells_total is None or >= len(dataset) all cells are returned
    directly without any sampling — this is the correct behaviour when the
    caller wants to use every available cell.
    """
    rng = np.random.default_rng(random_state)
    n_total = len(dataset)

    # None means no limit — return all cells immediately
    if max_cells_total is None or max_cells_total >= n_total:
        print(f'   Patient-aware sampling: returning all {n_total} cells (no limit).')
        return list(range(n_total)), {'all_cells': n_total}

    if donor_key not in dataset.column_names:
        n = min(max_cells_total, n_total)
        return rng.choice(n_total, size=n, replace=False).tolist(), {'all_cells': n}

    donors = dataset[donor_key]
    donor_to_indices = {}
    for i, d in enumerate(donors):
        donor_to_indices.setdefault(d, []).append(i)
    eligible = {d: idxs for d, idxs in donor_to_indices.items()
                if len(idxs) >= min_cells_per_donor}
    if not eligible:
        n = min(max_cells_total, n_total)
        print(f'   ⚠️  No donors with ≥{min_cells_per_donor} cells — random sampling ({n} cells).')
        return rng.choice(n_total, size=n, replace=False).tolist(), {'all_cells': n}

    n_donors = len(eligible)
    cells_per_donor = max(min_cells_per_donor, int(np.floor(max_cells_total / n_donors)))
    sampled, report = [], {}
    for donor, idxs in eligible.items():
        n_take = min(cells_per_donor, len(idxs))
        chosen = rng.choice(idxs, size=n_take, replace=False).tolist()
        sampled.extend(chosen)
        report[donor] = n_take

    # Only downsample further if we overshot the budget
    if len(sampled) > max_cells_total:
        sampled = rng.choice(sampled, size=max_cells_total, replace=False).tolist()

    print(f'   Patient-aware sampling: {len(sampled)} cells from {n_donors} donors '
          f'(~{cells_per_donor} cells/donor).')
    return sampled, report


def trim_sequences_safe(dataset, max_len=1024, protected_token_ids=None):
    if protected_token_ids is None:
        protected_token_ids = set()
    lengths  = dataset['length']
    max_orig = int(np.max(lengths))
    if max_len >= max_orig:
        return dataset
    sample_ids    = dataset['input_ids'][0]
    cls_token     = int(sample_ids[0])
    eos_token     = int(sample_ids[-1])
    fully_protected = protected_token_ids | {cls_token, eos_token}
    def trim_example(example):
        ids = list(example['input_ids'])
        if len(ids) <= max_len:
            return example
        cls    = ids[0]
        eos    = ids[-1]
        middle = ids[1:-1]
        target_middle = max_len - 2
        if len(middle) <= target_middle:
            return example
        keep_middle = middle[:target_middle]
        tail_middle = middle[target_middle:]
        protected_in_tail = [tid for tid in tail_middle if int(tid) in protected_token_ids]
        if protected_in_tail:
            swap_positions = [i for i in range(len(keep_middle)-1, -1, -1)
                              if int(keep_middle[i]) not in fully_protected]
            for swap_pos, prot_tid in zip(swap_positions, protected_in_tail):
                keep_middle[swap_pos] = prot_tid
        example['input_ids'] = [cls] + keep_middle + [eos]
        example['length']    = len(example['input_ids'])
        return example
    return dataset.map(trim_example, num_proc=1)


def score_one_state(cell_type, state_label, candidate_df, candidate_token_ids_set,
                    merged_by_state, embeddings, emb_feature_cols):
    if state_label not in merged_by_state:
        print(f'   [{state_label}] Not in pickles — skipping.')
        return None
    state_dict = merged_by_state[state_label]
    ctrl_mask  = ((embeddings['cell_type'] == cell_type) &
                  (embeddings['condition'] == CONTROL_LABEL))
    state_mask = ((embeddings['cell_type'] == cell_type) &
                  (embeddings['condition'] == state_label))
    ctrl_mean  = embeddings.loc[ctrl_mask, emb_feature_cols].values.mean(axis=0)
    state_mean = (embeddings.loc[state_mask, emb_feature_cols].values.mean(axis=0)
                  if state_mask.sum() > 0 else ctrl_mean)
    ctrl_norm  = ctrl_mean  / (np.linalg.norm(ctrl_mean)  + 1e-12)
    state_norm = state_mean / (np.linalg.norm(state_mean) + 1e-12)
    baseline_cos_sim = float(np.dot(ctrl_norm, state_norm))
    _sample_key = next(((tid, ek) for (tid, ek) in state_dict.keys() if ek == 'cell_emb'), None)
    if _sample_key:
        _raw = np.array(state_dict[_sample_key][:10], dtype=np.float64)
        _raw = _raw[np.isfinite(_raw)]
        if len(_raw) > 0 and abs(float(np.median(_raw))) < 0.1:
            baseline_cos_sim = 0.0
    token_to_ensembl = {v: k for k, v in tk.gene_token_dict.items()}
    all_gene_shifts  = {}
    all_gene_n_cells = {}
    rng_score = np.random.default_rng(42)
    for (token_id, emb_key), cos_sims in state_dict.items():
        if emb_key != 'cell_emb':
            continue
        arr = np.array(cos_sims, dtype=np.float64)
        arr = arr[np.isfinite(arr)]
        all_gene_n_cells[token_id] = len(arr)
        if len(arr) >= MIN_CELLS_PER_GENE:
            all_gene_shifts[token_id] = arr - baseline_cos_sim
    candidate_gene_shifts = {tid: s for tid, s in all_gene_shifts.items() if tid in candidate_token_ids_set}
    other_gene_shifts     = {tid: s for tid, s in all_gene_shifts.items() if tid not in candidate_token_ids_set}
    if not candidate_gene_shifts and not candidate_token_ids_set:
        return None
    if other_gene_shifts:
        other_pool = np.concatenate(list(other_gene_shifts.values()))
        other_pool = other_pool[np.isfinite(other_pool)]
    else:
        other_pool = (np.concatenate(list(candidate_gene_shifts.values()))
                      if candidate_gene_shifts else np.zeros(10))
        other_pool = other_pool[np.isfinite(other_pool)]
    rows = []
    for token_id in candidate_token_ids_set:
        ensembl_id  = token_to_ensembl.get(token_id, str(token_id))
        match       = candidate_df.loc[candidate_df['ensembl_id'] == ensembl_id, 'gene_symbol']
        gene_symbol = match.values[0] if len(match) > 0 else ensembl_id
        n_cells     = all_gene_n_cells.get(token_id, 0)
        if token_id in candidate_gene_shifts:
            shifts_a = candidate_gene_shifts[token_id]
            n_a = len(shifts_a)
            other_for_this = np.concatenate([v for tid, v in other_gene_shifts.items()
                                              if tid != token_id]) if other_gene_shifts else other_pool
            other_for_this = other_for_this[np.isfinite(other_for_this)]
            if len(other_for_this) >= n_a:
                sample_b = rng_score.choice(other_for_this, size=n_a, replace=False)
            elif len(other_for_this) > 0:
                sample_b = rng_score.choice(other_for_this, size=n_a, replace=True)
            else:
                sample_b = np.zeros(n_a)
            if np.median(sample_b) < 0:
                sample_b = np.maximum(sample_b, 0.0)
            try:
                _, pval = stats.ranksums(shifts_a, sample_b)
            except Exception:
                pval = 1.0
            if not np.isfinite(pval):
                pval = 1.0
            median_shift = float(np.nanmedian(shifts_a))
            mean_shift   = float(np.nanmean(shifts_a))
            std_shift    = float(np.nanstd(shifts_a))
        else:
            pval = 1.0
            median_shift = mean_shift = std_shift = 0.0
        rows.append({
            'gene_symbol': gene_symbol, 'ensembl_id': ensembl_id,
            'median_cosine_shift': median_shift, 'mean_cosine_shift': mean_shift,
            'std_cosine_shift': std_shift,
            'median_cos_sim': float(median_shift + baseline_cos_sim),
            'mean_cos_sim':   float(mean_shift   + baseline_cos_sim),
            'n_cells': n_cells, 'pval_raw': float(pval),
            'cell_type': cell_type, 'control_state': CONTROL_LABEL,
            'target_state': state_label, 'perturb_mode': PERTURB_MODE,
            'baseline_cos_sim': baseline_cos_sim,
        })
    if not rows:
        return None
    df = pd.DataFrame(rows)
    df['median_cosine_shift'] = df['median_cosine_shift'].fillna(0.0)
    df['mean_cosine_shift']   = df['mean_cosine_shift'].fillna(0.0)
    df['pval_raw']            = df['pval_raw'].fillna(1.0)
    _, pval_adj, _, _ = multipletests(df['pval_raw'], alpha=0.05, method='fdr_bh')
    df['pval_adj'] = pval_adj
    if gf_perturb_type == 'delete':
        df['significant'] = ((df['pval_adj'] < 0.05) &
                             (df['median_cosine_shift'].abs() > EFFECT_SIZE_THRESHOLD) &
                             #(df['median_cosine_shift'] < 0) &
                             (df['n_cells'] >= MIN_CELLS_PER_GENE))
    else:
        df['significant'] = ((df['pval_adj'] < 0.05) &
                             (df['median_cosine_shift'].abs() > EFFECT_SIZE_THRESHOLD) &
                             #(df['median_cosine_shift'] > 0) &
                             (df['n_cells'] >= MIN_CELLS_PER_GENE))
    df.loc[df['n_cells'] < MIN_CELLS_PER_GENE, 'significant'] = False
    ascending = (gf_perturb_type == 'delete')
    df = df.sort_values('median_cosine_shift', ascending=ascending).reset_index(drop=True)
    return df


print('✅ Helper functions defined.')

In [ ]:
import sys, os, time, hashlib, shutil, tempfile, subprocess, pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import loompy
import datasets as hf_datasets
import mygene
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from collections import defaultdict
from scipy import stats
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import gseapy as gp

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8,
})

if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

from geneformer import TranscriptomeTokenizer, EmbExtractor, InSilicoPerturber
import geneformer.perturber_utils as pu

mg = mygene.MyGeneInfo()

# GPU
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['TOKENIZERS_PARALLELISM']   = 'false'
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU — ISP will be slow.')

# Patch pu.load_model for GPU (once per session)
if not getattr(pu, '_load_model_gpu_patched', False):
    _REAL_LOAD_MODEL = pu.load_model
    def _gpu_load_model(model_type, num_classes, model_directory, *args, **kwargs):
        model = _REAL_LOAD_MODEL(model_type, num_classes, model_directory, *args, **kwargs)
        if torch.cuda.is_available():
            model = model.to('cuda:0').eval()
            print(f'   Model device: {next(model.parameters()).device}')
        return model
    pu.load_model = _gpu_load_model
    pu._load_model_gpu_patched = True
    print('✅ pu.load_model patched for GPU.')

# Constants
MODE_MAP        = {'down': 'delete', 'delete': 'delete', 'up': 'overexpress'}
gf_perturb_type = MODE_MAP.get(PERTURB_MODE, 'delete')
MIN_CELLS_PER_GENE    = 20
EFFECT_SIZE_THRESHOLD = 2.5e-4
N_BOOTSTRAP_SPLITS    = 10
SPLIT_FRACTION        = 0.5
MIN_CELLS_HALF        = 5
_alt_states           = ALT_STATES if 'ALT_STATES' in dir() and ALT_STATES else []
states_to_run         = [TARGET_LABEL] + list(_alt_states)

# Colour scheme
SIG_GREEN         = '#2ecc71'
FAIL_RED          = '#e74c3c'
SIG_COLOR         = '#C0392B'
NSIG_COLOR        = '#BDC3C7'
ALT_COLOR         = '#2980B9'
PAN_DISEASE_COLOR = '#8E44AD'
GOAL_LIGHT        = '#E8A49C'
ALT_LIGHT         = '#A9C4D9'
AMBER             = '#F5A623'

# Accumulators (filled during the loop, consumed in Step 14)
all_results            = []   # list of scored DataFrames
all_sep_rows           = []   # list of separation test summary rows
separation_results_all = {}   # {cell_type: {comp_name: result_dict}}
passing_celltypes      = []   # cell types that passed all sep tests
embeddings_all         = []   # list of per-cell-type embedding DataFrames

print('✅ Shared setup complete.')

# ════════════════════════════════════════════════════════════════════════════
# PER-CELL-TYPE LOOP
# Each iteration covers Steps 4–13 for one cell type.
# ════════════════════════════════════════════════════════════════════════════

# Output sub-directories (created once)
TOKENIZED_DIR     = os.path.join(OUTPUT_DIR, 'tokenized')
EMB_DIR           = os.path.join(OUTPUT_DIR, 'embeddings')
SEP_FIG_DIR       = os.path.join(OUTPUT_DIR, 'separation_test_figures')
ISP_OUTPUT_DIR    = os.path.join(OUTPUT_DIR, 'isp_output')
STATS_OUTPUT_DIR  = os.path.join(OUTPUT_DIR, 'isp_stats')
ISP_FIG_DIR       = os.path.join(OUTPUT_DIR, 'isp_figures')
for d in [TOKENIZED_DIR, EMB_DIR, SEP_FIG_DIR, ISP_OUTPUT_DIR, STATS_OUTPUT_DIR, ISP_FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# Tokenizer (built once, reused across cell types)
tk = TranscriptomeTokenizer(
    custom_attr_name_dict={'_cell_type': 'cell_type', '_condition': 'condition',
                           '_assay_type': 'assay_type', '_patient_id': 'patient_id'},
    nproc=4
)

# ISP write helper
original_write = pu.write_perturbation_dictionary
def safe_write(data, path):
    basename = os.path.basename(path)
    if len(basename) > 200:
        dir_part    = os.path.dirname(path)
        safe_prefix = basename[:60]
        token_hash  = hashlib.md5(basename.encode()).hexdigest()[:16]
        path        = os.path.join(dir_part, f'{safe_prefix}_{token_hash}')
    original_write(data, path)
pu.write_perturbation_dictionary = safe_write

# PatchedISP class (identical to v4)
class PatchedISP(InSilicoPerturber):
    """Thin wrapper around InSilicoPerturber.

    All cell/gene filtering is done by the caller before handing off
    `input_data_file`, so this class just ensures `genes_to_perturb='all'`
    and delegates directly to the base class via an unambiguous explicit call.

    Crucially we do NOT override `perturb_data` with `super().perturb_data()`
    because in Python's MRO that resolves back to this subclass and causes
    infinite recursion.  Instead we call `InSilicoPerturber.perturb_data`
    (the concrete base implementation) explicitly.
    """
    def __init__(self, **kwargs):
        kwargs['genes_to_perturb'] = 'all'
        super().__init__(**kwargs)


# ── MAIN LOOP ────────────────────────────────────────────────────────────────
for cell_type, adata_ct in adata_by_celltype.items():
    safe_ct = cell_type.replace(' ', '_').replace('/', '_')
    print(f'\n{"═"*65}')
    print(f'CELL TYPE: {cell_type}  ({adata_ct.n_obs:,} cells)')
    print(f'{"═"*65}')

    ct_out = os.path.join(OUTPUT_DIR, safe_ct)
    os.makedirs(ct_out, exist_ok=True)

    # ── Step 4: Gene mapping ─────────────────────────────────────────────
    print(f'\n[{cell_type}] Step 4: Gene mapping...')
    gene_names = adata_ct.var_names.tolist()
    results_mg = mg.querymany(
        gene_names,
        scopes='symbol' if GENE_ID_TYPE == 'symbol' else 'ensembl.gene',
        fields='ensembl.gene', species='human', returnall=False,
        as_dataframe=True, verbose=False
    )
    def extract_ensembl(row):
        val = row.get('ensembl.gene', np.nan)
        if isinstance(val, list):
            return val[0]
        return val
    results_mg['ensembl_id'] = results_mg.apply(extract_ensembl, axis=1)
    gene_map = results_mg['ensembl_id'].dropna().to_dict()
    adata_ct.var['ensembl_id'] = adata_ct.var_names.map(gene_map)
    n_mapped = adata_ct.var['ensembl_id'].notna().sum()
    print(f'   Mapped {n_mapped:,}/{len(gene_names):,} genes to Ensembl IDs.')
    adata_ct = adata_ct[:, adata_ct.var['ensembl_id'].notna()].copy()
    if adata_ct.n_vars == 0:
        print(f'   ⚠️  No genes mapped for {cell_type} — skipping.')
        continue

    # ── Step 5: Tokenize ─────────────────────────────────────────────────
    print(f'\n[{cell_type}] Step 5: Tokenizing...')
    ct_tok_dir  = os.path.join(TOKENIZED_DIR, safe_ct)
    ct_tok_data = os.path.join(OUTPUT_DIR, 'tokenized_data', safe_ct)
    os.makedirs(ct_tok_dir, exist_ok=True)
    os.makedirs(ct_tok_data, exist_ok=True)

    X = adata_ct.layers['counts'].copy()
    if sp.issparse(X):
        X = X.toarray()
    X = X.astype(np.float32)
    ensembl_ids = adata_ct.var['ensembl_id'].values.astype(str)
    n_counts    = X.sum(axis=1).astype(np.float32)
    matrix      = X.T
    row_attrs = {'ensembl_id': ensembl_ids, 'gene_name': ensembl_ids}
    _assay_arr   = (adata_ct.obs['assay_type'].values.tolist()
                    if 'assay_type' in adata_ct.obs.columns
                    else ['unknown'] * adata_ct.n_obs)
    _patient_arr = (adata_ct.obs['patient_id'].values.tolist()
                    if 'patient_id' in adata_ct.obs.columns
                    else ['unknown'] * adata_ct.n_obs)
    col_attrs = {
        'CellID'     : np.array(adata_ct.obs_names.tolist()),
        'n_counts'   : n_counts,
        '_cell_type' : np.array(adata_ct.obs['_cell_type'].values.tolist()),
        '_condition' : np.array(adata_ct.obs['_condition'].values.tolist()),
        '_assay_type': np.array(_assay_arr),
        '_patient_id': np.array(_patient_arr),
    }
    loom_path = os.path.join(ct_tok_dir, f'{safe_ct}.loom')
    loompy.create(loom_path, matrix, row_attrs, col_attrs)
    print(f'   Loom saved: {loom_path}')

    tk.tokenize_data(
        data_directory=ct_tok_dir,
        output_directory=ct_tok_data,
        output_prefix=f'geneformer_{safe_ct}',
        file_format='loom'
    )
    DATASET_PATH_CT = os.path.join(ct_tok_data, f'geneformer_{safe_ct}.dataset')
    print(f'   Tokenized dataset: {DATASET_PATH_CT}')

    # Sanitise cell type names in token dataset
    tok_dataset = hf_datasets.load_from_disk(DATASET_PATH_CT)
    tok_dataset = tok_dataset.map(
        lambda ex: {**ex, 'cell_type': ex['cell_type'].replace('/', '_')}, num_proc=1
    ).flatten_indices()

    # Fix: Save to a new temporary path to avoid PermissionError
    temp_dataset_path = os.path.join(ct_tok_data, f'geneformer_{safe_ct}_temp.dataset')
    tok_dataset.save_to_disk(temp_dataset_path)
    # Update DATASET_PATH_CT to the new temporary path
    DATASET_PATH_CT = temp_dataset_path
    print(f'   Sanitized dataset saved to: {DATASET_PATH_CT}')

    # ── Step 6: Extract embeddings ────────────────────────────────────────
    print(f'\n[{cell_type}] Step 6: Extracting embeddings...')
    ct_emb_dir  = os.path.join(EMB_DIR, safe_ct)
    os.makedirs(ct_emb_dir, exist_ok=True)

    embex = EmbExtractor(
        model_type='Pretrained', num_classes=0, emb_mode='cell',
        cell_emb_style='mean_pool',
        # filter_data={'cell_type': [safe_ct]}, # Removed redundant 'cell_type' filter
        max_ncells=None, emb_layer=-1,
        emb_label=['cell_type', 'condition', 'patient_id'], nproc=4,
    )
    embeddings = embex.extract_embs(
        model_directory='ctheodoris/Geneformer',
        input_data_file=DATASET_PATH_CT,
        output_directory=ct_emb_dir,
        output_prefix=f'embs_{safe_ct}',
    )
    emb_save_path = os.path.join(ct_emb_dir, f'embs_{safe_ct}.parquet')
    embeddings.to_parquet(emb_save_path, index=True)
    embeddings_all.append(embeddings)
    print(f'   Embeddings: {embeddings.shape}')

    emb_meta_cols    = {'cell_type', 'condition', 'cell_id', 'donor', 'patient_id'}
    emb_feature_cols = [c for c in embeddings.columns
                        if c not in emb_meta_cols
                        and pd.api.types.is_numeric_dtype(embeddings[c])]

    _donor_col_in_emb = None
    for _dc in ['donor', 'Donor', 'donor_id', 'patient_id', 'sample_id']:
        if _dc and _dc in embeddings.columns:
            _donor_col_in_emb = _dc
            break

    # ── Step 7: Separation test ───────────────────────────────────────────
    print(f'\n[{cell_type}] Step 7: Separation test...')
    ct_sep_results = {}

    for label_a, label_b, comp_name in state_pairs:
        has_a = ((embeddings['cell_type'] == cell_type) &
                 (embeddings['condition'] == label_a)).sum() >= MIN_CELLS_PER_STATE
        has_b = ((embeddings['cell_type'] == cell_type) &
                 (embeddings['condition'] == label_b)).sum() >= MIN_CELLS_PER_STATE
        if not has_a or not has_b:
            print(f'   {cell_type} [{comp_name}]: missing condition data — skipping.')
            ct_sep_results[comp_name] = {
                'passes': False, 'p_value': 1.0, 'sep_score': 0.0,
                'cohens_d': None, 'balanced_accuracy': 0.5,
                'n_ctrl': 0, 'n_tgt': 0, 'n_donors_ctrl': 0, 'n_donors_tgt': 0,
                'ctrl_margins': np.array([]), 'tgt_margins': np.array([])
            }
            continue
        res = run_separation_test_publishable(
            embeddings, cell_type, label_a, label_b,
            emb_cols=emb_feature_cols, donor_col=_donor_col_in_emb
        )
        ct_sep_results[comp_name] = res
        d_str  = f"{res['cohens_d']:.3f}" if res['cohens_d'] is not None else 'N/A'
        status = '✅ PASS' if res['passes'] else '❌ FAIL'
        print(f'   [{comp_name}]: {status}  sep={res["sep_score"]:.4f}  '
              f'p={res["p_value"]:.2e}  d={d_str}  '
              f'bal_acc={res["balanced_accuracy"]:.3f}  '
              f'n=({res["n_ctrl"]},{res["n_tgt"]})')
        all_sep_rows.append({
            'cell_type': cell_type, 'comparison': comp_name,
            'label_a': label_a, 'label_b': label_b,
            'passes': res['passes'],
            'sep_score': round(res['sep_score'], 5),
            'p_value': res['p_value'],
            'cohens_d': round(res['cohens_d'], 3) if res['cohens_d'] is not None else None,
            'balanced_accuracy': round(res['balanced_accuracy'], 3),
            'n_a': res['n_ctrl'], 'n_b': res['n_tgt'],
            'n_donors_a': res['n_donors_ctrl'], 'n_donors_b': res['n_donors_tgt'],
        })

    separation_results_all[cell_type] = ct_sep_results

    # Save per-cell-type separation results
    ct_sep_df = pd.DataFrame([
        r for r in all_sep_rows if r['cell_type'] == cell_type
    ])
    ct_sep_df.to_csv(os.path.join(ct_out, f'separation_test_{safe_ct}.csv'), index=False)

    # ── Separation test gate ──────────────────────────────────────────────
    ct_passes_sep = all_comparisons_pass(ct_sep_results)
    if ct_passes_sep:
        passing_celltypes.append(cell_type)
        print(f'\n   ✅ {cell_type} passes all separation tests → proceeding to ISP.')
    else:
        print(f'\n   ❌ {cell_type} FAILED separation test → skipping ISP.')
        print('      (Results will still appear in cross-cell-type summary plots.)')
        continue   # ← skip Steps 8–13 for this cell type

    # ── Step 8: Candidate gene selection ─────────────────────────────────
    print(f'\n[{cell_type}] Step 8: Building candidate gene list...')
    sym_to_ensembl = dict(zip(adata_ct.var_names, adata_ct.var['ensembl_id']))

    import pickle as _pkl
    vocab_path = '/content/Geneformer/geneformer/gene_median_dictionary.pkl'
    gf_vocab = None
    if os.path.exists(vocab_path):
        with open(vocab_path, 'rb') as f:
            gf_vocab = set(_pkl.load(f).keys())

    rows_cand = []
    for symbol in MASH_GENES:
        eid = sym_to_ensembl.get(symbol)
        if eid is None or (isinstance(eid, float) and np.isnan(eid)):
            continue
        if gf_vocab and eid not in gf_vocab:
            continue
        token_id = tk.gene_token_dict.get(eid)
        if token_id is None:
            continue
        rows_cand.append({'gene_symbol': symbol, 'ensembl_id': eid, 'token_id': token_id})

    candidate_df = pd.DataFrame(rows_cand)
    print(f'   {len(candidate_df)} candidate genes mapped.')
    candidate_df.to_csv(os.path.join(ct_out, f'candidate_genes_{safe_ct}.csv'), index=False)

    if len(candidate_df) == 0:
        print(f'   ⚠️  No candidate genes for {cell_type} — skipping ISP.')
        continue

    candidate_token_ids = [int(row.token_id) for row in candidate_df.itertuples()]

    # ── Step 9: In-Silico Perturbation ────────────────────────────────────
    print(f'\n[{cell_type}] Step 9: Running ISP...')
    ct_isp_dir = os.path.join(ISP_OUTPUT_DIR, safe_ct)
    os.makedirs(ct_isp_dir, exist_ok=True)

    # Load & filter tokenised dataset to this cell type
    full_dataset    = hf_datasets.load_from_disk(DATASET_PATH_CT)
    ct_cond_dataset = full_dataset.filter(
        lambda x: x['condition'] == CONTROL_LABEL,
        num_proc=1
    )
    print(f'   {len(ct_cond_dataset)} control cells.')

    _donor_key_isp = None
    for _dk in ['donor', 'Donor', 'donor_id', 'patient_id', 'sample_id']:
        if _dk and _dk in ct_cond_dataset.column_names:
            _donor_key_isp = _dk
            break

    # Patient-aware sampling
    if _donor_key_isp and len(ct_cond_dataset) > 0:
        sampled_idx, donor_report = patient_aware_sample_for_isp(
            ct_cond_dataset, donor_key=_donor_key_isp,
            max_cells_total=MAX_ISP_CELLS_TOTAL, min_cells_per_donor=MIN_CELLS_PER_DONOR_ISP,
        )
        pd.DataFrame(list(donor_report.items()), columns=['donor', 'n_cells']).to_csv(
            os.path.join(ct_isp_dir, f'patient_sampling_{safe_ct}.csv'), index=False
        )
    else:
        rng_isp = np.random.default_rng(42)
        n_take  = len(ct_cond_dataset) if MAX_ISP_CELLS_TOTAL is None else min(MAX_ISP_CELLS_TOTAL, len(ct_cond_dataset))
        sampled_idx  = list(range(n_take)) if n_take == len(ct_cond_dataset) else rng_isp.choice(len(ct_cond_dataset), size=n_take, replace=False).tolist()
        donor_report = {'random_fallback': n_take}

    isp_input_dataset = ct_cond_dataset.select(sampled_idx)
    candidate_set_int = set(int(t) for t in candidate_token_ids)
    isp_input_dataset = isp_input_dataset.filter(
        lambda x: any(int(t) in candidate_set_int for t in x['input_ids']), num_proc=1
    )
    print(f'   {len(isp_input_dataset)} cells after candidate filter.')

    if len(isp_input_dataset) == 0:
        print(f'   ⚠️  No cells with candidate genes — skipping ISP for {cell_type}.')
        continue

    isp_input_dataset = trim_sequences_safe(
        isp_input_dataset, max_len=1024, protected_token_ids=candidate_set_int
    ).flatten_indices()

    isp_tmp_dir    = tempfile.mkdtemp(prefix='isp_gpu_')
    isp_input_path = os.path.join(isp_tmp_dir, 'isp_input.dataset')
    isp_input_dataset.save_to_disk(isp_input_path)

    # State embeddings
    ctrl_mask = ((embeddings['cell_type'] == cell_type) &
                 (embeddings['condition'] == CONTROL_LABEL))
    tgt_mask  = ((embeddings['cell_type'] == cell_type) &
                 (embeddings['condition'] == TARGET_LABEL))
    control_embs = embeddings.loc[ctrl_mask, emb_feature_cols].values
    target_embs  = embeddings.loc[tgt_mask,  emb_feature_cols].values
    state_embs_dict = {
        CONTROL_LABEL: torch.tensor(control_embs.mean(axis=0), dtype=torch.float32).to(DEVICE),
        TARGET_LABEL:  torch.tensor(target_embs.mean(axis=0),  dtype=torch.float32).to(DEVICE),
    }
    for alt_state in _alt_states:
        alt_mask = ((embeddings['cell_type'] == cell_type) &
                    (embeddings['condition'] == alt_state))
        if alt_mask.sum() > 0:
            state_embs_dict[alt_state] = torch.tensor(
                embeddings.loc[alt_mask, emb_feature_cols].values.mean(axis=0),
                dtype=torch.float32
            ).to(DEVICE)

    try:
        # Input dataset is already pre-filtered (control cells only,
        # candidate-gene-containing cells only, trimmed to 1024 tokens).
        # filter_data=None avoids a second internal pass that would try to
        # match 'cell_type' strings and drop all cells.
        # InSilicoPerturber.perturb_data is called directly (no override)
        # to avoid the super() → self recursion that occurred in v4.
        isp = PatchedISP(
            perturb_type=gf_perturb_type, perturb_rank_shift=None,
            combos=0, anchor_gene=None,
            model_type='Pretrained', num_classes=0,
            emb_mode='cls', cell_emb_style='mean_pool',
            filter_data=None,
            cell_states_to_model={
                'state_key':   'condition',
                'start_state': CONTROL_LABEL,
                'goal_state':  TARGET_LABEL,
                'alt_states':  _alt_states
            },
            state_embs_dict=state_embs_dict,
            max_ncells=None, emb_layer=-1,
            forward_batch_size=64, nproc=1,
        )
        InSilicoPerturber.perturb_data(
            isp,
            model_directory='ctheodoris/Geneformer',
            input_data_file=isp_input_path,
            output_directory=ct_isp_dir,
            output_prefix=f'isp_{safe_ct}'
        )
        print(f'   ✅ ISP complete for {cell_type}.')
    finally:
        shutil.rmtree(isp_tmp_dir, ignore_errors=True)
        torch.cuda.empty_cache()

    # ── Step 10: Cosine shift & stats ─────────────────────────────────────
    print(f'\n[{cell_type}] Step 10: Computing cosine shifts...')
    ct_stats_dir = os.path.join(STATS_OUTPUT_DIR, safe_ct)
    os.makedirs(ct_stats_dir, exist_ok=True)

    pickle_files = [
        f for f in os.listdir(ct_isp_dir)
        if f.endswith('.pickle') and 'dict_cell_embs_' in f
    ]
    print(f'   {len(pickle_files)} pickle files found.')

    if not pickle_files:
        print('   No pickle files — skipping stats for this cell type.')
        continue

    merged_by_state = defaultdict(lambda: defaultdict(list))
    for fname in pickle_files:
        try:
            with open(os.path.join(ct_isp_dir, fname), 'rb') as f:
                batch_dict = pickle.load(f)
            if isinstance(batch_dict, list):
                batch_dict = batch_dict[0] if batch_dict else {}
            for sl, inner_dict in batch_dict.items():
                for key, val in inner_dict.items():
                    if isinstance(val, list):
                        merged_by_state[sl][key].extend(val)
                    else:
                        merged_by_state[sl][key].append(val)
        except Exception as e:
            print(f'   Failed to load {fname}: {e}')

    candidate_token_ids_set = {
        tk.gene_token_dict[eid]
        for eid in candidate_df['ensembl_id'].dropna()
        if eid in tk.gene_token_dict
    }

    for state_label in states_to_run:
        df_state = score_one_state(
            cell_type, state_label, candidate_df,
            candidate_token_ids_set, merged_by_state,
            embeddings, emb_feature_cols
        )
        if df_state is None:
            continue
        # Override cell_type to use the display name
        df_state['cell_type'] = cell_type
        safe_state = state_label.replace(' ', '_')
        out_csv    = os.path.join(ct_stats_dir, f'stats_{safe_ct}_{safe_state}.csv')
        df_state.to_csv(out_csv, index=False)
        all_results.append(df_state)
        n_sig = df_state['significant'].sum()
        print(f'   [{state_label}] {len(df_state)} genes scored, {n_sig} significant.')

    # ── Step 11: Rank & summarize (per-cell-type printout) ────────────────
    print(f'\n[{cell_type}] Step 11: Summary...')
    for state_label in states_to_run:
        ct_state_results = [df for df in all_results
                            if df['cell_type'].iloc[0] == cell_type
                            and df['target_state'].iloc[0] == state_label]
        if not ct_state_results:
            continue
        ct_df_state = ct_state_results[-1]
        ct_sig = ct_df_state[ct_df_state['significant']]
        gene_col = 'gene_symbol' if 'gene_symbol' in ct_df_state.columns else 'ensembl_id'
        print(f'   [{state_label}] {len(ct_sig)} significant hits:')
        if len(ct_sig) > 0:
            print(ct_sig[[gene_col, 'median_cosine_shift', 'pval_adj', 'n_cells']].head(10).to_string())

    # ── Step 12: Per-cell-type figures ────────────────────────────────────
    print(f'\n[{cell_type}] Step 12: Saving per-cell-type figures...')
    ct_fig_dir = os.path.join(ISP_FIG_DIR, safe_ct)
    os.makedirs(ct_fig_dir, exist_ok=True)

    # Only use results for this cell type
    ct_all_results = [df for df in all_results if df['cell_type'].iloc[0] == cell_type]
    if not ct_all_results:
        print(f'   No results to plot for {cell_type}.')
        continue

    results_df_ct = pd.concat(ct_all_results, ignore_index=True)
    shift_col = 'median_cosine_shift'
    padj_col  = 'pval_adj'
    gene_col  = 'gene_symbol' if 'gene_symbol' in results_df_ct.columns else 'ensembl_id'

    goal_df_ct = results_df_ct[results_df_ct['target_state'] == TARGET_LABEL].copy()

    # Volcano + bar + distribution for this cell type
    if not goal_df_ct.empty:
        goal_df_ct['-log10_padj'] = -np.log10(goal_df_ct[padj_col].clip(lower=1e-300))
        n_sig = goal_df_ct['significant'].sum()
        fig, axes = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={'width_ratios': [2, 1.5, 1.5]})
        fig.suptitle(
            f'ISP: {cell_type}\n{CONTROL_LABEL} → {TARGET_LABEL}  |  '
            f'Mode: {PERTURB_MODE}  |  n={n_sig} significant',
            fontsize=12, fontweight='bold', y=1.02
        )
        # Panel A: Volcano
        ax = axes[0]
        colors_map = goal_df_ct['significant'].map({True: SIG_COLOR, False: NSIG_COLOR})
        ax.scatter(goal_df_ct[shift_col], goal_df_ct['-log10_padj'],
                   c=colors_map, alpha=0.7, s=20, linewidths=0, rasterized=True)
        ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.6)
        ax.axhline(-np.log10(0.05), color='#7F8C8D', lw=0.8, ls=':', alpha=0.7)
        top_sig = goal_df_ct[goal_df_ct['significant']].nsmallest(8, shift_col)
        for _, row in top_sig.iterrows():
            ax.annotate(str(row.get(gene_col, '')),
                        xy=(row[shift_col], row['-log10_padj']),
                        xytext=(4, 4), textcoords='offset points', fontsize=7.5,
                        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
        ax.set_xlabel('Median Cosine Shift', fontsize=11)
        ax.set_ylabel('-log₁₀(FDR p-value)', fontsize=11)
        ax.set_title('A. Volcano', fontsize=11, fontweight='bold', loc='left')
        ax.legend(handles=[
            mpatches.Patch(color=SIG_COLOR, label=f'Significant (n={n_sig})'),
            mpatches.Patch(color=NSIG_COLOR, label='Not significant'),
        ], fontsize=9)
        # Panel B: Ranked bar
        ax2 = axes[1]
        top20 = goal_df_ct.nsmallest(20, shift_col).copy()
        bar_colors = [SIG_COLOR if s else NSIG_COLOR for s in top20['significant']]
        labels = top20[gene_col].tolist()
        ax2.barh(range(len(top20)), top20[shift_col].values[::-1],
                 color=bar_colors[::-1], edgecolor='none', height=0.7)
        ax2.set_yticks(range(len(top20)))
        ax2.set_yticklabels(labels[::-1], fontsize=9)
        ax2.axvline(0, color='black', lw=0.8)
        ax2.set_xlabel('Median Cosine Shift', fontsize=10)
        ax2.set_title('B. Top 20 Hits', fontsize=11, fontweight='bold', loc='left')
        # Panel C: Distribution
        ax3 = axes[2]
        sig_shifts  = goal_df_ct.loc[goal_df_ct['significant'],  shift_col].dropna()
        nsig_shifts = goal_df_ct.loc[~goal_df_ct['significant'], shift_col].dropna()
        if len(sig_shifts) > 1 and sig_shifts.nunique() > 1:
            ax3.hist(sig_shifts.values, bins=min(20, len(sig_shifts)),
                     color=SIG_COLOR, alpha=0.7, label=f'Sig (n={len(sig_shifts)})', density=True)
        if len(nsig_shifts) > 1 and nsig_shifts.nunique() > 1:
            ax3.hist(nsig_shifts.values, bins=min(20, len(nsig_shifts)),
                     color=NSIG_COLOR, alpha=0.5, label=f'Not sig (n={len(nsig_shifts)})', density=True)
        ax3.axvline(0, color='black', lw=0.8, ls='--')
        ax3.set_xlabel('Median Cosine Shift', fontsize=10)
        ax3.set_ylabel('Density', fontsize=10)
        ax3.set_title('C. Distribution', fontsize=11, fontweight='bold', loc='left')
        ax3.legend(fontsize=8)
        plt.tight_layout()
        fig_path = os.path.join(ct_fig_dir, f'isp_goal_{safe_ct}.pdf')
        plt.savefig(fig_path, dpi=300, bbox_inches='tight', format='pdf')
        plt.savefig(fig_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
        plt.show()
        print(f'   Saved: {fig_path}')

    # ── Step 13: Pathway enrichment ───────────────────────────────────────
    print(f'\n[{cell_type}] Step 13: Pathway enrichment...')
    for state_label in states_to_run:
        state_df_ct = results_df_ct[
            results_df_ct['target_state'] == state_label
        ] if 'target_state' in results_df_ct.columns else results_df_ct
        ct_sig_genes = state_df_ct.loc[
            state_df_ct['significant'], gene_col
        ].dropna().unique().tolist()
        if len(ct_sig_genes) < 5:
            print(f'   [{state_label}] Fewer than 5 significant genes — skipping enrichment.')
            continue
        enr_outdir = os.path.join(ct_out, f'enrichment_{state_label.replace(" ", "_")}')
        os.makedirs(enr_outdir, exist_ok=True)
        try:
            enr = gp.enrichr(
                gene_list=ct_sig_genes,
                gene_sets=['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023'],
                organism='human', outdir=enr_outdir, cutoff=0.05, no_plot=False
            )
            top_paths = enr.results.nsmallest(10, 'Adjusted P-value')[
                ['Term', 'Adjusted P-value', 'Overlap', 'Genes']
            ]
            print(top_paths.to_string())
            top_paths.to_csv(
                os.path.join(ct_out, f'enrichment_{safe_ct}_{state_label.replace(" ", "_")}_top10.csv'),
                index=False
            )
        except Exception as e:
            print(f'   Enrichment failed [{state_label}]: {e}')

    print(f'\n✅ Finished all steps for: {cell_type}')

# Restore original write function
pu.write_perturbation_dictionary = original_write

print(f'\n{"═"*65}')
print(f'Per-cell-type loop complete.')
print(f'Passing cell types ({len(passing_celltypes)}): {passing_celltypes}')
print(f'Total ISP result rows: {sum(len(df) for df in all_results)}')


## 14. Cross-Cell-Type Summary Plots

These plots aggregate results across all passing cell types and are generated after the loop.

In [ ]:
# ── Separation test summary (all cell types) ──────────────────────────────────
if all_sep_rows:
    sep_summary = pd.DataFrame(all_sep_rows)
    sep_summary_path = os.path.join(OUTPUT_DIR, 'separation_test_summary.csv')
    sep_summary.to_csv(sep_summary_path, index=False)
    print('Separation Test Summary (all cell types):')
    display(sep_summary[['cell_type', 'comparison', 'passes', 'sep_score',
                          'p_value', 'cohens_d', 'balanced_accuracy',
                          'n_a', 'n_b']].sort_values(['cell_type', 'comparison']).to_string(index=False))
else:
    print('No separation test results collected.')

In [ ]:
# ── Cross-cell-type ISP results ───────────────────────────────────────────────
if not all_results:
    print('No ISP results to summarize.')
else:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df = results_df.dropna(subset=['median_cosine_shift'])
    results_df['pval_adj'] = results_df['pval_adj'].fillna(1.0)

    final_path = os.path.join(STATS_OUTPUT_DIR, 'all_isp_results.csv')
    results_df.to_csv(final_path, index=False)
    print(f'All results saved: {final_path}  ({len(results_df)} rows)')

    gene_col  = 'gene_symbol' if 'gene_symbol' in results_df.columns else 'ensembl_id'
    shift_col = 'median_cosine_shift'
    padj_col  = 'pval_adj'
    goal_df   = results_df[results_df['target_state'] == TARGET_LABEL].copy()

    # ── Dot plot: goal state across cell types ────────────────────────────
    if len(passing_celltypes) >= 2:
        all_sig = goal_df[goal_df['significant']].dropna(subset=[shift_col]).copy()
        if len(all_sig) > 0:
            top_genes = (all_sig.groupby(gene_col)[shift_col].mean()
                         .nsmallest(20).index.tolist())
            pivot_shift = goal_df[goal_df[gene_col].isin(top_genes)].pivot_table(
                index=gene_col, columns='cell_type', values=shift_col, aggfunc='mean'
            ).fillna(0)
            pivot_padj = goal_df[goal_df[gene_col].isin(top_genes)].pivot_table(
                index=gene_col, columns='cell_type', values=padj_col, aggfunc='mean'
            ).fillna(1.0)
            x_labels = pivot_shift.columns.tolist()
            y_labels = pivot_shift.index.tolist()
            fig_dot, ax_dot = plt.subplots(
                figsize=(max(6, len(x_labels)*2.5), max(6, len(y_labels)*0.5))
            )
            for xi, ct in enumerate(x_labels):
                for yi, gene in enumerate(y_labels):
                    pv  = pivot_padj.loc[gene, ct] if ct in pivot_padj.columns else 1.0
                    sz  = max(20, -np.log10(pv + 1e-300) * 15)
                    col = SIG_COLOR if pv < 0.05 else NSIG_COLOR
                    ax_dot.scatter(xi, yi, s=sz, c=[col], alpha=0.85,
                                   linewidths=0.5, edgecolors='black')
            ax_dot.set_xticks(range(len(x_labels)))
            ax_dot.set_xticklabels(x_labels, rotation=35, ha='right', fontsize=9)
            ax_dot.set_yticks(range(len(y_labels)))
            ax_dot.set_yticklabels(y_labels, fontsize=9)
            ax_dot.set_xlabel('Cell Type', fontsize=11)
            ax_dot.set_ylabel('Gene', fontsize=11)
            ax_dot.set_title(
                f'Top Perturbation Hits Across Cell Types — Control → {TARGET_LABEL}\n'
                'Dot size ∝ −log₁₀(FDR p-value)  |  Red = significant',
                fontsize=11, fontweight='bold'
            )
            ax_dot.grid(True, alpha=0.2, linewidth=0.5)
            plt.tight_layout()
            dot_path = os.path.join(ISP_FIG_DIR, 'isp_dotplot_ctrl_vs_goal.pdf')
            plt.savefig(dot_path, dpi=300, bbox_inches='tight', format='pdf')
            plt.savefig(dot_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
            plt.show()
            print(f'Saved: {dot_path}')

    # ── Alt state dot plots ───────────────────────────────────────────────
    for alt_state in _alt_states:
        alt_df = results_df[results_df['target_state'] == alt_state].copy()
        if alt_df.empty or len(passing_celltypes) < 2:
            continue
        all_sig_alt = alt_df[alt_df['significant']].dropna(subset=[shift_col])
        if len(all_sig_alt) == 0:
            print(f'No significant hits for {alt_state} — dot plot skipped.')
            continue
        top_genes_alt = (all_sig_alt.groupby(gene_col)[shift_col].mean()
                         .nsmallest(20).index.tolist())
        pivot_shift_alt = alt_df[alt_df[gene_col].isin(top_genes_alt)].pivot_table(
            index=gene_col, columns='cell_type', values=shift_col, aggfunc='mean'
        ).fillna(0)
        pivot_padj_alt = alt_df[alt_df[gene_col].isin(top_genes_alt)].pivot_table(
            index=gene_col, columns='cell_type', values=padj_col, aggfunc='mean'
        ).fillna(1.0)
        x_labels_alt = pivot_shift_alt.columns.tolist()
        y_labels_alt = pivot_shift_alt.index.tolist()
        fig_dot_alt, ax_dot_alt = plt.subplots(
            figsize=(max(6, len(x_labels_alt)*2.5), max(6, len(y_labels_alt)*0.5))
        )
        for xi, ct in enumerate(x_labels_alt):
            for yi, gene in enumerate(y_labels_alt):
                pv  = pivot_padj_alt.loc[gene, ct] if ct in pivot_padj_alt.columns else 1.0
                sz  = max(20, -np.log10(pv + 1e-300) * 15)
                col = ALT_COLOR if pv < 0.05 else NSIG_COLOR
                ax_dot_alt.scatter(xi, yi, s=sz, c=[col], alpha=0.85,
                                   linewidths=0.5, edgecolors='black')
        ax_dot_alt.set_xticks(range(len(x_labels_alt)))
        ax_dot_alt.set_xticklabels(x_labels_alt, rotation=35, ha='right', fontsize=9)
        ax_dot_alt.set_yticks(range(len(y_labels_alt)))
        ax_dot_alt.set_yticklabels(y_labels_alt, fontsize=9)
        ax_dot_alt.set_xlabel('Cell Type', fontsize=11)
        ax_dot_alt.set_ylabel('Gene', fontsize=11)
        ax_dot_alt.set_title(
            f'Top Perturbation Hits Across Cell Types — Control → {alt_state}\n'
            'Dot size ∝ −log₁₀(FDR p-value)  |  Blue = significant',
            fontsize=11, fontweight='bold'
        )
        ax_dot_alt.grid(True, alpha=0.2, linewidth=0.5)
        plt.tight_layout()
        dot_alt_path = os.path.join(
            ISP_FIG_DIR, f'isp_dotplot_ctrl_vs_{alt_state.replace(" ", "_")}.pdf'
        )
        plt.savefig(dot_alt_path, dpi=300, bbox_inches='tight', format='pdf')
        plt.savefig(dot_alt_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
        plt.show()
        print(f'Saved: {dot_alt_path}')

print('\n✅ Cross-cell-type summary plots complete.')

## 15. Export Final Results

In [ ]:
import zipfile
from google.colab import files

zip_path = '/content/drive/MyDrive/scFM/geneformer_isp_results_v5.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, '/content'))

print(f'✅ All results zipped: {zip_path}')
files.download(zip_path)
print('\nKey output files per cell type (under geneformer_isp_results/<CellType>/):')
print('  candidate_genes_<ct>.csv          — genes tested')
print('  separation_test_<ct>.csv          — separation test result')
print('  isp_stats/<ct>/stats_<ct>_*.csv   — cosine shift scores')
print('  isp_figures/<ct>/isp_goal_<ct>.pdf — per-cell-type ISP figure')
print('  enrichment_*/                      — pathway enrichment')
print('\nCross-cell-type outputs (under geneformer_isp_results/):')
print('  separation_test_summary.csv        — all cell types')
print('  isp_stats/all_isp_results.csv      — all ISP results combined')
print('  isp_figures/isp_dotplot_*.pdf      — cross-cell-type dot plots')